# Notebook 8 — April 2026: Persona-Based Recommendation Simulation

**The question:** In April, which customer personas drive the most revenue, and what should the app recommend to each?

This notebook runs the recommendation engine from Notebook 5 across all 5 personas under April conditions, producing a **concrete recommendation playbook** for store managers and the mobile app team.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
from IPython.display import display, HTML

np.random.seed(42)

ON_KAGGLE = Path('/kaggle/working').exists()
if ON_KAGGLE:
    DATA_DIR = Path('/kaggle/input/starbucks-recommendation-engine')
    if not DATA_DIR.exists():
        DATA_DIR = Path('/kaggle/input/datasets/shiratoriseto/starbucks-recommendation-engine')
else:
    DATA_DIR = Path('../data')

menu = pd.read_csv((DATA_DIR / 'raw' if not ON_KAGGLE else DATA_DIR) / 'starbucks_menu.csv')
txn = pd.read_csv((DATA_DIR / 'processed' if not ON_KAGGLE else DATA_DIR) / 'synthetic_transactions.csv', parse_dates=['date'])

print(f'Menu: {menu.shape} | Transactions: {txn.shape}')

## Section 1 — April Persona Mix

Who walks in during April?

In [ ]:
# ── April persona analysis ────────────────────────────────────────────
april_txn = txn[txn['date'].dt.month == 4].copy()

persona_mix = april_txn['persona'].value_counts(normalize=True).sort_values(ascending=True)
persona_revenue = april_txn.groupby('persona')['total_price'].agg(['mean', 'sum', 'count'])
persona_revenue['revenue_share'] = persona_revenue['sum'] / persona_revenue['sum'].sum()
persona_revenue = persona_revenue.round(3)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh(persona_mix.index, persona_mix.values * 100, color='#00704A', edgecolor='white')
axes[0].set_title('April Persona Mix (%)', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Share (%)')
for i, v in enumerate(persona_mix.values):
    axes[0].text(v * 100 + 0.3, i, f'{v:.1%}', va='center', fontsize=9)

axes[1].barh(persona_revenue.index, persona_revenue['revenue_share'] * 100, color='#E76F51', edgecolor='white')
axes[1].set_title('April Revenue Share by Persona (%)', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Revenue Share (%)')
for i, (idx, row) in enumerate(persona_revenue.iterrows()):
    axes[1].text(row['revenue_share'] * 100 + 0.3, i, f'{row["revenue_share"]:.1%} (avg ${row["mean"]:.2f})', va='center', fontsize=9)

plt.tight_layout()
plt.show()

print('\n=== April Persona Summary ===')
display(persona_revenue)

## Section 2 — Recommendation Engine

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# RECOMMENDATION ENGINE (from Notebook 5, self-contained)
# ══════════════════════════════════════════════════════════════════════

PERSONA_CONSTRAINTS = {
    'morning_commuter': {'description': 'Needs caffeine, quick, moderate budget',
        'max_calories': 400, 'min_caffeine_mg': 75, 'max_price': 6.00,
        'preferred_categories': ['Coffee', 'Classic Espresso Drinks'],
        'time_slots': ['early_morning', 'morning_rush'], 'customization_budget': 1.50},
    'afternoon_treat': {'description': 'Indulgent, willing to spend',
        'max_calories': 600, 'min_caffeine_mg': 0, 'max_price': 7.50,
        'preferred_categories': ['Frappuccino® Blended Coffee', 'Frappuccino® Blended Crème', 'Signature Espresso Drinks'],
        'time_slots': ['lunch', 'afternoon'], 'customization_budget': 2.50},
    'health_conscious': {'description': 'Low calorie, low sugar',
        'max_calories': 200, 'min_caffeine_mg': 0, 'max_price': 6.50,
        'preferred_categories': ['Coffee', 'Tazo® Tea Drinks', 'Frappuccino® Light Blended Coffee'],
        'time_slots': ['morning_rush', 'late_morning'], 'customization_budget': 1.00},
    'student': {'description': 'Budget-conscious, needs energy',
        'max_calories': 500, 'min_caffeine_mg': 50, 'max_price': 5.50,
        'preferred_categories': ['Frappuccino® Blended Coffee', 'Shaken Iced Beverages', 'Classic Espresso Drinks'],
        'time_slots': ['late_morning', 'afternoon', 'evening'], 'customization_budget': 1.00},
    'weekend_explorer': {'description': 'Adventurous, tries new things',
        'max_calories': 500, 'min_caffeine_mg': 0, 'max_price': 7.00,
        'preferred_categories': ['Signature Espresso Drinks', 'Smoothies', 'Frappuccino® Blended Coffee'],
        'time_slots': ['late_morning', 'lunch', 'afternoon'], 'customization_budget': 2.00},
}

CUSTS = [
    {'name': 'Extra Shot', 'price': 0.80, 'calories': 5, 'caffeine': 75},
    {'name': 'Vanilla Syrup', 'price': 0.60, 'calories': 20, 'caffeine': 0},
    {'name': 'Caramel Drizzle', 'price': 0.60, 'calories': 15, 'caffeine': 0},
    {'name': 'Oat Milk', 'price': 0.70, 'calories': 10, 'caffeine': 0},
    {'name': 'Cold Foam', 'price': 1.00, 'calories': 35, 'caffeine': 0},
    {'name': 'Sugar-Free Vanilla', 'price': 0.60, 'calories': 0, 'caffeine': 0},
]

COLD_CATS = ['Frappuccino® Blended Coffee', 'Frappuccino® Blended Crème',
             'Frappuccino® Light Blended Coffee', 'Shaken Iced Beverages', 'Smoothies']

def score_item(item, persona, temp, hour):
    c = PERSONA_CONSTRAINTS[persona]
    if item['calories'] > c['max_calories']: return -1, []
    if item['caffeine_mg'] < c['min_caffeine_mg']: return -1, []
    if item['price_usd'] > c['max_price']: return -1, []
    
    score = 10
    reasons = []
    if item['category'] in c['preferred_categories']:
        score += 30; reasons.append('Preferred category')
    if temp > 70 and item['category'] in COLD_CATS:
        score += 20; reasons.append('Cold drink for warm weather')
    elif temp < 55 and item['category'] not in COLD_CATS:
        score += 20; reasons.append('Hot drink for cool weather')
    else:
        score += 10
    
    ts = 'morning_rush' if 7<=hour<10 else 'afternoon' if 14<=hour<17 else 'lunch' if 12<=hour<14 else 'evening'
    if ts in c['time_slots']: score += 15; reasons.append('Good time fit')
    if item['seasonal_flag'] == 1: score += 15; reasons.append('Seasonal special')
    return score, reasons

def recommend(persona, temp, hour, top_n=3):
    scored = []
    for _, item in menu.iterrows():
        s, r = score_item(item, persona, temp, hour)
        if s > 0:
            scored.append({'product_name': item['product_name'], 'category': item['category'],
                'size': item['size'], 'price': item['price_usd'], 'calories': item['calories'],
                'caffeine_mg': item['caffeine_mg'], 'score': s, 'reasons': r})
    scored.sort(key=lambda x: -x['score'])
    seen = set()
    unique = []
    for s in scored:
        if s['product_name'] not in seen:
            seen.add(s['product_name']); unique.append(s)
    return unique[:top_n]

print('Engine ready.')

## Section 3 — April Recommendation Playbook

For each persona, we run the recommender at April-typical conditions and produce a **concrete recommendation card**.

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# APRIL RECOMMENDATION PLAYBOOK
# ══════════════════════════════════════════════════════════════════════

april_scenarios = {
    'morning_commuter': {'temp': 55, 'hour': 8, 'context': 'Weekday 8AM, 55°F (cool spring morning)'},
    'afternoon_treat':  {'temp': 65, 'hour': 14, 'context': 'Weekday 2PM, 65°F (warming up)'},
    'health_conscious': {'temp': 60, 'hour': 10, 'context': 'Weekday 10AM, 60°F (mild morning)'},
    'student':          {'temp': 62, 'hour': 16, 'context': 'Weekday 4PM, 62°F (study session)'},
    'weekend_explorer': {'temp': 68, 'hour': 11, 'context': 'Saturday 11AM, 68°F (spring brunch)'},
}

all_recs = []
for persona, scenario in april_scenarios.items():
    print('\n' + '=' * 65)
    print(f'  {persona.upper().replace("_", " ")}')
    print(f'  {PERSONA_CONSTRAINTS[persona]["description"]}')
    print(f'  Context: {scenario["context"]}')
    print('=' * 65)
    
    recs = recommend(persona, scenario['temp'], scenario['hour'])
    for rank, rec in enumerate(recs, 1):
        print(f'\n  #{rank} {rec["product_name"]} ({rec["size"]})')
        print(f'      ${rec["price"]:.2f} | {rec["calories"]} cal | {rec["caffeine_mg"]}mg caffeine')
        print(f'      Score: {rec["score"]} — {" | ".join(rec["reasons"])}')
        all_recs.append({**rec, 'persona': persona, 'rank': rank})

recs_df = pd.DataFrame(all_recs)

## Section 4 — Revenue Impact: April Upsell Potential

In [ ]:
# ── Upsell analysis by persona for April ──────────────────────────────

upsell_results = []
for persona, scenario in april_scenarios.items():
    recs = recommend(persona, scenario['temp'], scenario['hour'], top_n=1)
    if recs:
        rec = recs[0]
        c = PERSONA_CONSTRAINTS[persona]
        budget = c['customization_budget']
        
        # Find applicable customizations
        custs_applied = []
        remaining = budget
        for cust in CUSTS:
            if cust['price'] <= remaining and rec['calories'] + cust['calories'] <= c['max_calories']:
                custs_applied.append(cust)
                remaining -= cust['price']
        
        upcharge = sum(c['price'] for c in custs_applied)
        upsell_results.append({
            'persona': persona,
            'base_product': rec['product_name'],
            'base_price': rec['price'],
            'customizations': ', '.join(c['name'] for c in custs_applied) if custs_applied else 'none',
            'upcharge': upcharge,
            'total_price': rec['price'] + upcharge,
            'upsell_pct': upcharge / rec['price'] * 100,
        })

upsell_df = pd.DataFrame(upsell_results)
display(upsell_df[['persona', 'base_product', 'base_price', 'customizations', 'upcharge', 'total_price', 'upsell_pct']].round(2))

print(f'\nAverage upsell: ${upsell_df["upcharge"].mean():.2f} ({upsell_df["upsell_pct"].mean():.1f}%)')
print(f'Average total ticket with customization: ${upsell_df["total_price"].mean():.2f}')

# Monthly impact
april_txn_count = len(april_txn)
monthly_uplift = april_txn_count * upsell_df['upcharge'].mean()
print(f'\nProjected April upsell revenue (if applied to all {april_txn_count:,} transactions): ${monthly_uplift:,.0f}')

In [ ]:
# ── Visualization: persona × recommendation matrix ────────────────────

fig = make_subplots(rows=1, cols=2, subplot_titles=('Top Recommendation by Persona', 'Upsell Potential'),
                    column_widths=[0.55, 0.45])

# Top picks
top_picks = recs_df[recs_df['rank'] == 1]
fig.add_trace(go.Bar(
    y=top_picks['persona'], x=top_picks['price'],
    orientation='h', marker_color='#00704A',
    text=[f"{r['product_name'][:25]} ${r['price']:.2f}" for _, r in top_picks.iterrows()],
    textposition='outside', textfont_size=9,
), row=1, col=1)

# Upsell
colors = ['#264653' if u > 0 else '#ccc' for u in upsell_df['upcharge']]
fig.add_trace(go.Bar(
    y=upsell_df['persona'], x=upsell_df['upsell_pct'],
    orientation='h', marker_color=colors,
    text=[f'+${u:.2f} ({p:.0f}%)' for u, p in zip(upsell_df['upcharge'], upsell_df['upsell_pct'])],
    textposition='outside', textfont_size=9,
), row=1, col=2)

fig.update_layout(title='April 2026: Personalized Recommendations & Upsell',
                  height=400, template='plotly_white', showlegend=False)
fig.update_xaxes(title_text='Base Price ($)', row=1, col=1)
fig.update_xaxes(title_text='Upsell %', row=1, col=2)
fig.show()

## Actionable Takeaways for April 2026

1. **Morning commuters** dominate volume — ensure espresso bar is fully staffed 7-9 AM
2. **Afternoon treat seekers** drive highest per-ticket revenue — push Frappuccino promotions in-app from 1 PM
3. **Students** are most price-sensitive — consider student discount or app-exclusive deal to increase visit frequency
4. **Weekend explorers** have highest customization potential — upsell via in-app "Try something new" prompts
5. **Health-conscious** segment is growing — ensure tea and light options are prominently displayed

**Key metric:** Personalized customization recommendations could add ~$0.50-$1.50 per transaction across all personas, translating to significant monthly revenue uplift.